# Hackology II — Trening w Google Colab

Ten notebook pokazuje jak wytrenować model YOLOv8 na danych hackathonowych
korzystając z darmowego GPU (T4) w Google Colab.

**GPU w Colab = GPU na ewaluacji** — środowisko prywatnej ewaluacji też używa NVIDIA T4.

## Wymagania

1. Konto Google
2. GitHub PAT (Personal Access Token) z dostępem do repo drużyny
3. Runtime ustawiony na **GPU** (Runtime → Change runtime type → T4 GPU)

## 0. Sprawdź GPU

Upewnij się, że masz GPU. Jeśli nie — zmień runtime: Runtime → Change runtime type → T4 GPU.

In [ ]:
!nvidia-smi

## 1. Setup — instalacja `uv` i zależności

In [ ]:
# Instalacja uv
!curl -LsSf https://astral.sh/uv/install.sh | sh

import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

!uv --version

## 2. Klonowanie repo drużyny

Wstaw swój GitHub PAT poniżej. Możesz też użyć Colab Secrets:
1. Kliknij ikonę 🔑 w lewym panelu
2. Dodaj secret `GITHUB_PAT` z wartością tokena
3. Włącz "Notebook access"

In [ ]:
import os

# --- KONFIGURACJA --- zmień te wartości na swoje
ORG = os.environ.get("HACKOLOGY_ORG", "hackology-2026")
TEAM_REPO = os.environ.get("HACKOLOGY_REPO", "team-XXX-detector")

# Kolejność: zmienna środowiskowa → Colab Secrets → manual input
if os.environ.get("GITHUB_PAT"):
    PAT = os.environ["GITHUB_PAT"]
    print("PAT załadowany z zmiennej środowiskowej")
else:
    try:
        from google.colab import userdata
        PAT = userdata.get("GITHUB_PAT")
        print("PAT załadowany z Colab Secrets")
    except Exception:
        PAT = input("Wklej GitHub PAT: ")

In [ ]:
import os

REPO_DIR = os.environ.get("HACKOLOGY_REPO_DIR", f"/content/{TEAM_REPO}")

if not os.path.exists(REPO_DIR):
    !git clone https://{PAT}@github.com/{ORG}/{TEAM_REPO}.git {REPO_DIR}
else:
    print(f"Repo już sklonowane w {REPO_DIR}")

os.chdir(REPO_DIR)
!pwd

In [ ]:
# Instalacja zależności z pyproject.toml
!uv sync

## 3. Pobranie danych

In [ ]:
!bash download_data.sh

## 4. Konwersja COCO → YOLO format

Ultralytics `model.train()` wymaga danych w formacie YOLO (pliki `.txt` z labelami).
Konwertujemy anotacje COCO na ten format.

In [ ]:
import json
from pathlib import Path


def coco_to_yolo(annotations_path: Path, images_dir: Path, labels_dir: Path, taxonomy_path: Path):
    """Konwertuj anotacje COCO na format YOLO (txt per obraz).

    Każdy plik .txt zawiera linie: class_idx x_center y_center width height
    (znormalizowane do [0, 1]).
    """
    labels_dir.mkdir(parents=True, exist_ok=True)

    with open(annotations_path) as f:
        coco = json.load(f)
    with open(taxonomy_path) as f:
        taxonomy = json.load(f)

    # Mapowanie category_id → indeks YOLO (0-based)
    cat_ids = sorted(c["id"] for c in taxonomy["categories"])
    cat_id_to_idx = {cid: idx for idx, cid in enumerate(cat_ids)}

    # Mapowanie image_id → info
    img_map = {img["id"]: img for img in coco["images"]}

    # Grupuj anotacje po image_id
    from collections import defaultdict
    anns_by_img = defaultdict(list)
    for ann in coco["annotations"]:
        anns_by_img[ann["image_id"]].append(ann)

    converted = 0
    for image_id, anns in anns_by_img.items():
        img_info = img_map[image_id]
        img_w = img_info["width"]
        img_h = img_info["height"]
        stem = Path(img_info["file_name"]).stem

        lines = []
        for ann in anns:
            cat_idx = cat_id_to_idx.get(ann["category_id"])
            if cat_idx is None:
                continue
            x, y, w, h = ann["bbox"]  # COCO: [x, y, w, h] (top-left)
            # YOLO: [x_center, y_center, w, h] (normalized)
            x_center = (x + w / 2) / img_w
            y_center = (y + h / 2) / img_h
            w_norm = w / img_w
            h_norm = h / img_h
            lines.append(f"{cat_idx} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

        label_file = labels_dir / f"{stem}.txt"
        label_file.write_text("\n".join(lines) + "\n")
        converted += 1

    print(f"Skonwertowano {converted} obrazów do {labels_dir}")


# Konwersja anotacji treningowych
coco_to_yolo(
    annotations_path=Path("data/train/annotations.json"),
    images_dir=Path("data/train/images"),
    labels_dir=Path("data/train/labels"),
    taxonomy_path=Path("taxonomy.json"),
)

In [ ]:
import json
from pathlib import Path

# Generuj dataset.yaml dla ultralytics (bez zależności od pyyaml)
with open("taxonomy.json") as f:
    taxonomy = json.load(f)

cat_ids = sorted(c["id"] for c in taxonomy["categories"])
names = {idx: next(c["name"] for c in taxonomy["categories"] if c["id"] == cid)
         for idx, cid in enumerate(cat_ids)}

lines = [
    f"path: {Path.cwd()}",
    "train: data/train/images",
    "val: data/train/images",  # brak osobnego val — używamy train do walidacji
    f"nc: {len(names)}",
    "names:",
]
for idx, name in sorted(names.items()):
    lines.append(f"  {idx}: {name}")

Path("dataset.yaml").write_text("\n".join(lines) + "\n")
print("Zapisano dataset.yaml:")
print("\n".join(lines))


## 5. Trening modelu

Fine-tuning YOLOv8 na danych hackathonowych.

Parametry można nadpisać zmiennymi środowiskowymi — przydatne przy testach lokalnych.

In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

# Parametry można nadpisać zmiennymi środowiskowymi (np. do testów lokalnych/CPU)
_device = os.environ.get("YOLO_DEVICE", "0")
try:
    _device = int(_device)
except ValueError:
    pass  # "cpu"

_epochs = int(os.environ.get("YOLO_EPOCHS", 10))
_imgsz  = int(os.environ.get("YOLO_IMGSZ", 640))
_batch  = int(os.environ.get("YOLO_BATCH", 16))
_model  = os.environ.get("YOLO_MODEL", "yolov8n.pt")

# Załaduj pretrenowany model
model = YOLO(_model)

# Fine-tuning na danych hackathonowych
results = model.train(
    data="dataset.yaml",
    epochs=_epochs,     # zwiększ dla lepszych wyników (30-100)
    imgsz=_imgsz,
    batch=_batch,       # T4 wytrzyma 16-32 przy imgsz=640
    device=_device,     # GPU (0) lub CPU ("cpu")
    project="runs",
    name="hackology",
    exist_ok=True,
)

WEIGHTS_PATH = str(Path(results.save_dir) / "weights" / "best.pt")
print(f"Wagi modelu: {WEIGHTS_PATH}")


## 6. Predykcja na zbiorze testowym

Uruchom `predict.py` na obrazach z `data/public_test/images/`. Wynik trafia do `submissions/predictions.json`.

In [ ]:
import os
from pathlib import Path

Path("submissions").mkdir(exist_ok=True)

# Użyj wag z treningu lub modelu bazowego gdy trening pominięty
_weights = WEIGHTS_PATH if "WEIGHTS_PATH" in dir() else os.environ.get("YOLO_MODEL", "yolov8n.pt")

!uv run predict \
    --input data/public_test/images \
    --output submissions/predictions.json \
    --model {_weights}


## 7. Walidacja formatu

Sprawdź, czy plik predykcji ma poprawny format przed wysłaniem.

In [ ]:
# Walidacja formatu
import json

with open("submissions/predictions.json") as f:
    preds = json.load(f)

print(f"Predykcji: {len(preds)}")
if preds:
    print(f"Przykład: {json.dumps(preds[0], indent=2)}")
    required_keys = {"image_id", "category_id", "bbox", "score"}
    missing = required_keys - set(preds[0].keys())
    if missing:
        print(f"BŁĄD: Brakuje kluczy: {missing}")
    else:
        print("Format OK")
else:
    print("UWAGA: Brak predykcji — model nic nie wykrył. Sprawdź taxonomy mapping.")


## 8. Submission — wyślij wynik

Commituj i pushuj predykcje. Workflow na GitHubie automatycznie wyśle zgłoszenie.

In [ ]:
!git config user.name "Colab"
!git config user.email "team@hackology.dev"
!git add submissions/predictions.json
!git commit -m "submission: yolov8 fine-tuned"
!git push


## Co dalej?

Pomysły na poprawę wyniku:

- **Więcej epok** — `epochs=50` lub `epochs=100`
- **Większy model** — `yolov8s.pt`, `yolov8m.pt`, `yolov8l.pt`
- **Augmentacja** — ultralytics ma wbudowane augmentacje, dodaj własne
- **Test Time Augmentation (TTA)** — `model.predict(..., augment=True)`
- **Ensemble** — wytrenuj kilka modeli i uśrednij predykcje
- **Confidence threshold** — eksperymentuj z `--confidence` w predict.py

Sprawdź notebook `01_exploration.ipynb` żeby lepiej zrozumieć dane.